In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score,roc_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

In [21]:
df = pd.read_csv('../data/processed/telco_churn_clean.csv')

In [22]:
df.head().T

,0,1,2,3,4
SeniorCitizen,0,0,0,0,0
Partner,1,0,0,0,0
Dependents,0,0,0,0,0
tenure,1,34,2,45,2
PaperlessBilling,1,0,1,0,1
Churn_Numeric,0,0,1,0,1
MultipleLines_No phone service,1,0,0,1,0
MultipleLines_Yes,0,0,0,0,0
InternetService_Fiber optic,0,0,0,0,1
InternetService_No,0,0,0,0,0


In [23]:
X = df.drop('Churn_Numeric',axis=1)
y = df['Churn_Numeric']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42,stratify=y)

scaler = StandardScaler()
X_train[['tenure']] = scaler.fit_transform(X_train[['tenure']])
X_test[['tenure']] = scaler.transform(X_test[['tenure']])

## Logistic Regression

In [42]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
modelSmote = LogisticRegression(random_state=42)
modelSmote.fit(X_train_smote, y_train_smote)
y_pred = modelSmote.predict(X_test)
print(f"Classification Report of the model with SMOTE:\n{classification_report(y_test, y_pred)}")
print("Confusion Matrix of the model with SMOTE:")
print(confusion_matrix(y_test, y_pred))
print("ROC AUC Score of the model with SMOTE:", roc_auc_score(y_test, modelSmote.predict_proba(X_test)[:, 1]))

print('---' * 20)

model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Classification Report of the model without SMOTE:\n{classification_report(y_test, y_pred)}")
print("Confusion Matrix of the model without SMOTE:")
print(confusion_matrix(y_test, y_pred))
print("ROC AUC Score of the model without SMOTE:", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))

Classification Report of the model with SMOTE:
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1552
           1       0.51      0.77      0.61       561

    accuracy                           0.74      2113
   macro avg       0.70      0.75      0.71      2113
weighted avg       0.79      0.74      0.75      2113

Confusion Matrix of the model with SMOTE:
[[1134  418]
 [ 129  432]]
ROC AUC Score of the model with SMOTE: 0.8317380138559641
------------------------------------------------------------
Classification Report of the model without SMOTE:
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1552
           1       0.51      0.79      0.62       561

    accuracy                           0.75      2113
   macro avg       0.71      0.76      0.72      2113
weighted avg       0.80      0.75      0.76      2113

Confusion Matrix of the model without SMOTE:
[[1133  419]
 [ 

In [25]:
# Define hyperparameter grid for Logistic Regression
param_dist_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [100, 200, 500, 1000],
    'class_weight': ['balanced', None]
}

# Perform RandomizedSearchCV
random_search_lr = RandomizedSearchCV(
    LogisticRegression(random_state=42),
    param_distributions=param_dist_lr,
    n_iter=20,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search_lr.fit(X_train, y_train)

# Get best model and parameters
best_lr_model = random_search_lr.best_estimator_
print("Best Parameters:", random_search_lr.best_params_)
print("Best CV Score:", random_search_lr.best_score_)

# Evaluate on test set
y_pred_best_lr = best_lr_model.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_lr))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_lr))
print("ROC AUC Score:", roc_auc_score(y_test, best_lr_model.predict_proba(X_test)[:, 1]))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters: {'solver': 'saga', 'penalty': 'l1', 'max_iter': 100, 'class_weight': 'balanced', 'C': 0.1}
Best CV Score: 0.6286269954301513

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1552
           1       0.51      0.79      0.62       561

    accuracy                           0.74      2113
   macro avg       0.71      0.76      0.71      2113
weighted avg       0.80      0.74      0.76      2113


Confusion Matrix:
[[1129  423]
 [ 117  444]]
ROC AUC Score: 0.8428472490214456


## XGBOOST

In [43]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

xg_model = XGBClassifier(random_state=42)
xg_model.fit(X_train,y_train)
y_pred_xg = xg_model.predict(X_test)
print(classification_report(y_test,y_pred_xg))
print(confusion_matrix(y_test,y_pred_xg))
print("ROC AUC Score:",roc_auc_score(y_test,y_pred_xg))

print('---' * 20)

xg_model_smote = XGBClassifier(random_state=42)
xg_model_smote.fit(X_train_smote,y_train_smote)
y_pred_xg_smote = xg_model_smote.predict(X_test)
print(classification_report(y_test,y_pred_xg_smote))
print(confusion_matrix(y_test,y_pred_xg_smote))
print("ROC AUC Score:",roc_auc_score(y_test,y_pred_xg_smote))

              precision    recall  f1-score   support

           0       0.83      0.87      0.85      1552
           1       0.58      0.51      0.54       561

    accuracy                           0.77      2113
   macro avg       0.70      0.69      0.69      2113
weighted avg       0.76      0.77      0.77      2113

[[1343  209]
 [ 275  286]]
ROC AUC Score: 0.6875694865575096
------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.86      0.79      0.82      1552
           1       0.52      0.64      0.58       561

    accuracy                           0.75      2113
   macro avg       0.69      0.71      0.70      2113
weighted avg       0.77      0.75      0.76      2113

[[1226  326]
 [ 203  358]]
ROC AUC Score: 0.7140473105830899


In [37]:
# Define hyperparameter grid for XGBoost
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.5, 1.0],
    # 'scale_pos_weight': [3,5,7]
}

# Perform RandomizedSearchCV
random_search = RandomizedSearchCV(
    XGBClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train_smote, y_train_smote)

# Get best model and parameters
best_xgb_model = random_search.best_estimator_
print("Best Parameters:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)

# Evaluate on test set
y_pred_best_xgb = best_xgb_model.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_xgb))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_xgb))
print("ROC AUC Score:", roc_auc_score(y_test, best_xgb_model.predict_proba(X_test)[:, 1]))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters: {'subsample': 0.9, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 9, 'learning_rate': 0.2, 'gamma': 0.1, 'colsample_bytree': 0.9}
Best CV Score: 0.9071488528727913

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.81      0.82      1552
           1       0.52      0.58      0.55       561

    accuracy                           0.74      2113
   macro avg       0.68      0.69      0.68      2113
weighted avg       0.75      0.74      0.75      2113


Confusion Matrix:
[[1250  302]
 [ 237  324]]
ROC AUC Score: 0.7869461749085764
